# Baseline GRN inference

Here, we use GRNBoost2 and GENIE3 as baseline methods for GRN inference against which we can compare the GRN reconstructed from bowside.
We use the 10 synthetic curated HSC datasets from BEELINE ([Pratapa et al. (2020)](https://doi.org/10.1038/s41592-019-0690-6)), downloaded from [Zenodo](https://doi.org/10.5281/zenodo.19009603) with the same expression matrices as used for bowside.

In [1]:
import nest_asyncio2

nest_asyncio2.apply()

import itertools
from pathlib import Path

import pandas as pd
import scanpy as sc
from arboreto.algo import genie3, grnboost2
from distributed import Client, LocalCluster
from tqdm.auto import tqdm

In [2]:
# Load AnnData object
dataset_names = ["GSD", "HSC", "mCAD", "VSC"]
adatas = {
    name: [
        sc.read_h5ad(
            f"../data/BEELINE-data/processed/Curated/{name}/{name}_processed_{i + 1}.h5ad"
        )
        for i in range(10)
    ]
    for name in dataset_names
}
for key, lst in adatas.items():
    print(key)
    display(lst[0])

GSD


AnnData object with n_obs × n_vars = 2000 × 19
    obs: 'PseudoTime1', 'PseudoTime2', 'pseudotime'
    var: 'mean', 'std'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    layers: 'log1p'
    obsp: 'connectivities', 'distances'

HSC


AnnData object with n_obs × n_vars = 2000 × 11
    obs: 'PseudoTime1', 'PseudoTime2', 'PseudoTime3', 'PseudoTime4', 'pseudotime'
    var: 'mean', 'std'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    layers: 'log1p'
    obsp: 'connectivities', 'distances'

mCAD


AnnData object with n_obs × n_vars = 2000 × 5
    obs: 'PseudoTime1', 'PseudoTime2', 'pseudotime'
    var: 'mean', 'std'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    layers: 'log1p'
    obsp: 'connectivities', 'distances'

VSC


AnnData object with n_obs × n_vars = 2000 × 8
    obs: 'PseudoTime1', 'PseudoTime2', 'PseudoTime3', 'PseudoTime4', 'PseudoTime5', 'pseudotime'
    var: 'mean', 'std'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    layers: 'log1p'
    obsp: 'connectivities', 'distances'

In [3]:
# Create output directories
out_dirs_grnboost = {
    name: Path(f"../output/BEELINE_{name}/grnboost2") for name in dataset_names
}
for out_dir in out_dirs_grnboost.values():
    out_dir.mkdir(parents=True, exist_ok=True)
out_dirs_genie = {
    name: Path(f"../output/BEELINE_{name}/genie3") for name in dataset_names
}
for out_dir in out_dirs_genie.values():
    out_dir.mkdir(parents=True, exist_ok=True)

In [4]:
exp_dfs = {
    name: [
        sc.get.obs_df(
            adata,
            keys=adata.var_names.tolist(),
            layer="log1p",
        )
        for adata in lst
    ]
    for name, lst in adatas.items()
}

In [5]:
local_cluster = LocalCluster()

In [6]:
# Run GRNBoost2 (multiple times)
n_runs = 10
grnboost_res_dict = {}

for name, exp_df_lst in tqdm(exp_dfs.items()):
    grnboost2_res_list = []
    for i, exp_df in tqdm(
        zip(range(n_runs), itertools.cycle(exp_df_lst)), total=10, leave=False
    ):
        custom_client = Client(local_cluster)
        grnboost2_res: pd.DataFrame = grnboost2(
            expression_data=exp_df,
            seed=i,
            client_or_address=custom_client,  # type: ignore
        )
        custom_client.close()
        grnboost2_res.to_parquet(out_dirs_grnboost[name] / f"grn_res_{i}.parquet")
        grnboost2_res_list.append(grnboost2_res)
    grnboost_res_dict[name] = grnboost2_res_list

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [7]:
# Run GENIE3 (multiple times)
genie_res_dict = {}

for name, exp_df_lst in tqdm(exp_dfs.items()):
    genie_res_list = []
    for i, exp_df in tqdm(
        zip(range(n_runs), itertools.cycle(exp_df_lst)), total=10, leave=False
    ):
        custom_client = Client(local_cluster)
        genie_res: pd.DataFrame = genie3(
            expression_data=exp_df,
            seed=i,
            client_or_address=custom_client,  # type: ignore
        )
        custom_client.close()
        genie_res.to_parquet(out_dirs_genie[name] / f"grn_res_{i}.parquet")
        genie_res_list.append(genie_res)
    genie_res_dict[name] = genie_res_list

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]